[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week04.ipynb)

# Week 4: Data Pipelines
**Introduction to Deep Learning (HIT)** &middot; Part II: Training Infrastructure

The Dataset and DataLoader abstractions; batching, shuffling, transforms, and splits.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Build a custom Dataset and DataLoader.
- Reason about batching, shuffling, and clean splits.
- Recognize and avoid data leakage.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. A custom Dataset
Implement __len__ and __getitem__ to return one (input, label).

In [ ]:
from torch.utils.data import Dataset, DataLoader

class ToyData(Dataset):
    def __init__(self, n=120):
        self.x = torch.randn(n, 3)
        self.y = (self.x.sum(1) > 0).long()
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

ds = ToyData()
print("len:", len(ds), "| one item:", ds[0][0].shape, ds[0][1].item())

## 2. A DataLoader batches and shuffles
Iterating yields batches shaped (batch, ...).

In [ ]:
dl = DataLoader(ds, batch_size=16, shuffle=True)
xb, yb = next(iter(dl))
print("batch:", tuple(xb.shape), tuple(yb.shape))
for i, (xb, yb) in enumerate(dl):
    print("batch", i, "->", xb.shape[0], "samples")
    if i == 3:
        break

## 3. Batch size and shuffling
Batch size sets the steps per epoch; shuffling reorders every epoch.

In [ ]:
for bs in [8, 16, 64]:
    print(f"batch_size={bs}: {len(DataLoader(ds, batch_size=bs))} batches per epoch")
dl = DataLoader(ds, batch_size=4, shuffle=True)
print("epoch 1 first labels:", next(iter(dl))[1].tolist())
print("epoch 2 first labels:", next(iter(dl))[1].tolist())

## 4. A train / validation / test split
Tune on validation, touch test once. random_split is the easy way.

In [ ]:
from torch.utils.data import random_split
g = torch.Generator().manual_seed(0)
tr, va, te = random_split(ds, [80, 20, 20], generator=g)
print("sizes:", len(tr), len(va), len(te))
train_dl = DataLoader(tr, batch_size=16, shuffle=True)
val_dl   = DataLoader(va, batch_size=16)            # no shuffle for val/test

## 5. Data leakage, demonstrated
Normalizing with whole-dataset statistics leaks test information.

In [ ]:
data = torch.randn(100, 1) * 5 + 10
train, test = data[:80], data[80:]

mu, sd = data.mean(), data.std()                 # LEAK: uses test data
print("leaky   test mean ~ 0:", round(((test - mu) / sd).mean().item(), 3))

mu, sd = train.mean(), train.std()               # correct: train only
print("correct test mean (not 0):", round(((test - mu) / sd).mean().item(), 3))

**Discuss:** name three other ways leakage sneaks in (feature selection on all data, time order ignored, duplicate rows across splits).

---
Student materials for this week: the lab handout (`labs/week04.html`) and the curated references (`references/week04.html`) in the course site.